In [0]:
df = spark.table("workspace.aqi.bronze_city_day")
display(df.limit(5))

City,Date,PM2.5,PM10,NO,NO2,NOx,NH3,CO,SO2,O3,Benzene,Toluene,Xylene,AQI,AQI_Bucket,_rescued_data
Ahmedabad,2015-01-01,null,null,0.92,18.22,17.15,null,0.92,27.64,133.36,0.0,0.02,0.0,null,null,null
Ahmedabad,2015-01-02,null,null,0.97,15.69,16.46,null,0.97,24.55,34.06,3.68,5.5,3.77,null,null,null
Ahmedabad,2015-01-03,null,null,17.4,19.3,29.7,null,17.4,29.07,30.7,6.8,16.4,2.25,null,null,null
Ahmedabad,2015-01-04,null,null,1.7,18.48,17.97,null,1.7,18.59,36.08,4.43,10.14,1.0,null,null,null
Ahmedabad,2015-01-05,null,null,22.1,21.42,37.76,null,22.1,39.33,39.31,7.01,18.89,2.78,null,null,null


In [0]:
from pyspark.sql.functions import col
df_silver = df.filter(col("AQI").isNotNull())
print(df_silver.count())

24850


In [0]:
df_silver = df_silver.select(
    col("City").alias("city"),
    col("Date").alias("date"),
    col("`PM2.5`").alias("pm25"),
    col("PM10").alias("pm10"),
    col("NO2").alias("no2"),
    col("SO2").alias("so2"),
    col("O3").alias("o3"),
    col("CO").alias("co"),
    col("NO").alias("no"),
    col("AQI").alias("aqi"),
    col("AQI_Bucket").alias("aqi_category")
)
display(df_silver.limit(5))

city,date,pm25,pm10,no2,so2,o3,co,no,aqi,aqi_category
Ahmedabad,2015-01-29,83.13,null,28.71,49.52,59.76,6.93,6.93,209.0,Poor
Ahmedabad,2015-01-30,79.84,null,28.68,48.49,97.07,13.85,13.85,328.0,Very Poor
Ahmedabad,2015-01-31,94.52,null,32.66,67.39,111.33,24.39,24.39,514.0,Severe
Ahmedabad,2015-02-01,135.99,null,42.08,75.23,102.7,43.48,43.48,782.0,Severe
Ahmedabad,2015-02-02,178.33,null,35.31,55.04,107.38,54.56,54.56,914.0,Severe


In [0]:
from pyspark.sql.functions import when

df_silver = df_silver.withColumn(
    "co",
    when((col("city") == "Ahmedabad") & (col("co") == col("no")), None)
    .otherwise(col("co"))
).drop("no")

display(df_silver.limit(5))

city,date,pm25,pm10,no2,so2,o3,co,aqi,aqi_category
Ahmedabad,2015-01-29,83.13,null,28.71,49.52,59.76,null,209.0,Poor
Ahmedabad,2015-01-30,79.84,null,28.68,48.49,97.07,null,328.0,Very Poor
Ahmedabad,2015-01-31,94.52,null,32.66,67.39,111.33,null,514.0,Severe
Ahmedabad,2015-02-01,135.99,null,42.08,75.23,102.7,null,782.0,Severe
Ahmedabad,2015-02-02,178.33,null,35.31,55.04,107.38,null,914.0,Severe


In [0]:
from pyspark.sql.functions import year, month

df_silver = (
    df_silver
    .withColumn("year", year(col("date")))
    .withColumn("month", month(col("date")))
    .withColumn("aqi_more_500", col("aqi") > 500)
)

display(df_silver.limit(5))

city,date,pm25,pm10,no2,so2,o3,co,aqi,aqi_category,year,month,aqi_more_500
Ahmedabad,2015-01-29,83.13,null,28.71,49.52,59.76,null,209.0,Poor,2015,1,false
Ahmedabad,2015-01-30,79.84,null,28.68,48.49,97.07,null,328.0,Very Poor,2015,1,false
Ahmedabad,2015-01-31,94.52,null,32.66,67.39,111.33,null,514.0,Severe,2015,1,true
Ahmedabad,2015-02-01,135.99,null,42.08,75.23,102.7,null,782.0,Severe,2015,2,true
Ahmedabad,2015-02-02,178.33,null,35.31,55.04,107.38,null,914.0,Severe,2015,2,true


In [0]:
df_silver = df_silver.withColumn(
    "season",
    when(col("month").isin(12, 1, 2), "Winter")
    .when(col("month").isin(3, 4, 5), "Summer")
    .when(col("month").isin(6, 7, 8, 9), "Monsoon")
    .otherwise("Post-Monsoon")
)

display(df_silver.groupBy("season").count())

season,count
Winter,6243
Summer,7069
Monsoon,7713
Post-Monsoon,3825


In [0]:
(
    df_silver.write
    .mode("overwrite")
    .partitionBy('year')
    .saveAsTable("workspace.aqi.silver_city_day_spark")
)
print("Saved!")

Saved!


In [0]:
display(spark.sql("DESCRIBE DETAIL workspace.aqi.silver_city_day_spark"))

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,438192db-77f4-4fd9-ae47-afe2d789d661,workspace.aqi.silver_city_day_spark,null,,2026-11-02T13:39:26.514Z,2026-11-02T13:39:31.000Z,List(year),List(),6,515799,"Map(delta.parquet.compression.codec -> zstd, delta.parquet.format.version.afe.internal -> 2.12.0, delta.enableDeletionVectors -> true, delta.parquet.format.version -> 2.12.0, delta.enableRowTracking -> true, delta.checkpointPolicy -> v2, io.unitycatalog.tableId -> 8e6ac398-1457-4aba-af92-f0650540fbb9, delta.rowTracking.materializedRowCommitVersionColumnName -> _row-commit-version-col-6b68d605-b5d3-47c4-8929-3ab918ac1321, delta.rowTracking.materializedRowIdColumnName -> _row-id-col-d833376e-5cbf-4130-bbd5-12b5b2975028)",3,7,"List(appendOnly, deletionVectors, domainMetadata, invariants, rowTracking, v2Checkpoint)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
sql_silver = spark.table("workspace.aqi.silver_city_day")
spark_silver = spark.table("workspace.aqi.silver_city_day_spark")

print("SQL silver rows: ", sql_silver.count())
print("PySpark silver rows: ", spark_silver.count())

cols = sql_silver.columns
difference = sql_silver.select(cols).exceptAll(spark_silver.select(cols))
print("Rows that differ: ", difference.count())

SQL silver rows:  24850
PySpark silver rows:  24850
Rows that differ:  0
